In [0]:
import pyspark.sql.functions as sf
from time import time
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

In [0]:
# How mmuch time should be analyzed in seconds
window_time = time() - 10000000

In [0]:
# Read data from 'runescape.02_silver.item_mapping'
df_map = spark.read.table("runescape.02_silver.item_mapping")

In [0]:
# Read data from 'runescape.01_bronze.1h_prices' to get Volumes
df_1hour = spark.read.table("runescape.01_bronze.1h_prices")

Enriched Data

In [0]:
# Read data from 'runescape.02_silver.1h_prices_enriched'
df_1h_prices_enriched = spark.read.table("runescape.02_silver.1h_prices_enriched")
df_1h_prices_enriched = df_1h_prices_enriched.drop("highalch","limit","members")

In [0]:
# get skill input data
df_skill_inputs_enriched = spark.read.table("runescape.02_silver.skill_action_inputs_enriched")

In [0]:
# get list of items that were traded in the last 2 weeks
# limit data to the last 2 weeks
two_week_time_window = 1209600
df_1h_prices_enriched_two_weeks = df_1h_prices_enriched.filter(f"time >= {two_week_time_window}")
df_1h_items = df_1h_prices_enriched_two_weeks.groupBy("id").count()
# get id's that are present in the table skill_action_inputs_enriched
# get id's if they are an output
df_1h_items_outputs = df_1h_items.join(df_skill_inputs_enriched
                                          , df_skill_inputs_enriched.id_output == df_1h_items.id
                                          , "inner").select("id")
# get id's if they are an input
df_1h_items_inputs = df_1h_items.join(df_skill_inputs_enriched
                                          , df_skill_inputs_enriched.id_input == df_1h_items.id
                                          , "inner").select("id")
# union outputs and inputs
df_1h_items_unioned = df_1h_items_outputs.union(df_1h_items_inputs).dropDuplicates()


# get list of times that have data for the last 2 weeks
df_1h_times = df_1h_prices_enriched_two_weeks.groupBy("time").count()
df_1h_times = df_1h_times.select("time")

In [0]:
# cross join df_1h_items_unioned and df_1h_times to get all combinations of items and times
df_items_times = df_1h_items_unioned.crossJoin(df_1h_times)

In [0]:
# join df_items_times with df_1h_prices_enriched
df_1h_prices = df_1h_prices_enriched.join(df_items_times, ["id", "time"], "right_outer")

In [0]:
# combine records with missing price data with data from 1h_prices_last to get imputed data for missing data
# get item time combination for missing data
df_items_times_missing = df_1h_prices.filter(sf.isnull("avg1HourHigh")).select("id","time")

In [0]:
# Read data from 'runescape.02_silver.1h_prices_last_enriched'
df_1h_prices_last_enriched = spark.read.table("runescape.02_silver.1h_prices_last_enriched")
# drop time column
df_1h_prices_last_enriched = df_1h_prices_last_enriched.drop("time", "highalch", "limit", "members")

In [0]:
# comnbine with data from 1h_prices_last_enriched
df_1h_prices_imputed = df_items_times_missing.join(df_1h_prices_last_enriched, "id")

In [0]:
# get real price data without Nulls
df_1h_prices_real = df_1h_prices.filter(sf.isnotnull("avg1HourHigh"))

# union 1h_price data to get complete data set
df_1h_prices_full = df_1h_prices_real.union(df_1h_prices_imputed)

Build table with profit calculations

In [0]:
#df_1h_skill_profit = df_1h_prices_full
# combine input data with price data
df_1h_prices_full_w_inputs = df_1h_prices_full.join(df_skill_inputs_enriched,
                                            df_1h_prices_full.id == df_skill_inputs_enriched.id_output,
                                            "left_outer")
df_1h_prices_full_w_inputs = df_1h_prices_full_w_inputs.drop('id','name','highalch','limit','members')
# rename columns to be associated with output data
df_1h_prices_full_w_inputs = df_1h_prices_full_w_inputs.withColumnsRenamed(
    {'avg1HourHigh':'avg_high_output',
     'avg1HourHighVolume':'avg_high_volume_output',
     'avg1HourLow':'avg_low_output',
     'avg1HourLowVolume':'avg_low_volume_output'}
)

In [0]:
# join to get input price data
df_1h_prices_full_combined = df_1h_prices_full_w_inputs.join(df_1h_prices_full,
                                                                     [df_1h_prices_full_w_inputs.id_input == df_1h_prices_full.id,
                                                                      df_1h_prices_full_w_inputs.time == df_1h_prices_full.time],
                                                                     "left_outer").drop(df_1h_prices_full_w_inputs.time)

In [0]:
# calculate input costs
df_input_cost = df_1h_prices_full_combined.withColumns({'input_cost_low': sf.round(sf.col("avg1HourLow")*sf.col("input_qty"),1),
                                                            'input_cost_high': sf.round(sf.col("avg1HourHigh")*sf.col("input_qty"),1)})

In [0]:
df_input_total_cost = df_input_cost.groupBy("time","output", "skill").sum("input_cost_high", "input_cost_low")
df_input_total_cost = df_input_total_cost.withColumnsRenamed({"sum(input_cost_high)": "total_input_cost_high"
                                                              , "sum(input_cost_low)": "total_input_cost_low"})


In [0]:
tax = .98 #simplified tax for now, can be implemented as function
df_1h_profits = (df_1h_prices_full_combined.join(df_input_total_cost, ["time", "output", "skill"])\
    .drop("name")\
    # TODO rename input cols
    # calculate profit estimates without tax
    # TODO add tax and implement as function
    .withColumns({
        "profit_low_est": sf.round(tax*sf.col("avg_low_output") - sf.col("total_input_cost_high"),2),
        "profit_high_est": sf.round(tax*sf.col("avg_high_output") - sf.col("total_input_cost_low"),2)
    }))